# Grammar and Style Improver

## 1. Project Overview

**Task:** Rewrite text for clarity, concision, and grammatical correctness using both rule-based checks and LLM-powered rewriting. We detect passive voice, wordiness, long sentences, and unclear phrasing, then improve them.

**Approach:** Three-layer improvement pipeline:
1. **Rule-based detection** — catch common issues with regex patterns
2. **LLM rewriting** — fix and improve in context
3. **Before/after evaluation** — score readability improvements

**Stack:** `LangChain` + `ChatOllama` + `qwen3.5:9b` — no API keys required.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Detect** passive voice, wordiness, and long sentences with rules |
| 2 | **Rewrite** for clarity using LLM prompts |
| 3 | **Combine** rule-based detection with LLM fixing |
| 4 | **Evaluate** readability improvements |
| 5 | Handle **different improvement goals** (concision, clarity, formality) |

## 3. Problem Statement

Most professional writing suffers from wordiness, passive constructions, and unclear structure. An automated improver can flag issues and suggest cleaner alternatives.

## 4. Why This Matters

- **Writing assistants** (Grammarly, Hemingway) are multi-billion dollar products
- **Rule + LLM hybrid** teaches how to combine deterministic and probabilistic approaches
- **Readability** directly impacts communication effectiveness in every profession

## 5. Setup

In [ ]:
import os, json, re, time, warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

LLM_MODEL = "qwen3.5:9b"
llm = ChatOllama(model=LLM_MODEL, temperature=0)

def clean(text):
    if "<think>" in text: text = text.split("</think>")[-1].strip()
    return text.strip()

def ask(system, user):
    return clean(llm.invoke([SystemMessage(content=system), HumanMessage(content=user)]).content)

print(f"LLM ready: {ask('Reply with one word.', 'Say ready.')[:20]}")

## 6. Sample Texts for Improvement

We use texts with known issues: wordiness, passive voice, long sentences, and unclear phrasing.

In [ ]:
SAMPLES = [
    {
        "id": 1,
        "text": "In order to improve our results, we need to increase the sample size due to the fact that it is currently too small to draw meaningful conclusions from the data that has been collected.",
        "issues": ["wordiness", "passive voice", "redundancy"],
    },
    {
        "id": 2,
        "text": "The report was written by the analyst and it was reviewed by the manager and then it was submitted to the client for their consideration at this point in time.",
        "issues": ["passive voice", "repetitive structure", "wordiness"],
    },
    {
        "id": 3,
        "text": "The data was processed and the model was trained using a complex pipeline that includes multiple steps of feature engineering and hyperparameter optimization and cross validation and ensemble methods.",
        "issues": ["run-on sentence", "passive voice", "no punctuation"],
    },
    {
        "id": 4,
        "text": "It should be noted that the implementation of the aforementioned system was undertaken in a manner that was consistent with the previously established guidelines that had been put in place by the committee.",
        "issues": ["extreme wordiness", "passive voice", "bureaucratic jargon"],
    },
    {
        "id": 5,
        "text": "There are many different factors that can potentially have an impact on the overall performance of the system in various different ways depending on the specific circumstances.",
        "issues": ["vague", "filler words", "no specifics"],
    },
    {
        "id": 6,
        "text": "I wanted to reach out to you in regards to the meeting that was scheduled for next week to discuss the project that we have been working on for the past several months in order to determine the next steps.",
        "issues": ["wordiness", "one giant sentence", "passive voice"],
    },
]

for s in SAMPLES:
    print(f"Sample {s['id']}: {s['text'][:80]}... | Issues: {', '.join(s['issues'])}")

---

## 7. Layer 1: Rule-Based Detection

Catch common writing issues with regex patterns.

In [ ]:
WORDY_PHRASES = {
    "in order to": "to",
    "due to the fact that": "because",
    "at this point in time": "now",
    "in the event that": "if",
    "for the purpose of": "to",
    "in spite of the fact that": "although",
    "it should be noted that": "(remove)",
    "in regards to": "about",
    "with respect to": "about",
    "the aforementioned": "the",
    "in a manner that was consistent with": "following",
    "can potentially have an impact on": "can affect",
    "various different": "various",
    "many different": "many",
    "overall performance": "performance",
    "specific circumstances": "circumstances",
    "previously established": "established",
    "I wanted to reach out to you": "I'm writing",
}

def detect_issues(text):
    issues = []
    # Passive voice detection
    passive_pattern = r'\b(was|were|been|being|is|are)\s+\w+ed\b'
    passive_matches = re.findall(passive_pattern, text, re.IGNORECASE)
    if passive_matches:
        issues.append(f"\ud83d\udfe1 Passive voice detected ({len(passive_matches)} instances)")
    
    # Wordy phrases
    for phrase, replacement in WORDY_PHRASES.items():
        if phrase.lower() in text.lower():
            issues.append(f"\ud83d\udfe0 Wordy: \"{phrase}\" \u2192 \"{replacement}\"")
    
    # Long sentences
    sentences = re.split(r'[.!?]+', text)
    for i, sent in enumerate(sentences):
        word_count = len(sent.split())
        if word_count > 25:
            issues.append(f"\ud83d\udd34 Sentence {i+1} too long ({word_count} words). Split it.")
    
    # Repeated "and" chains
    and_count = text.lower().count(" and ")
    if and_count >= 3:
        issues.append(f"\ud83d\udfe3 Too many 'and' conjunctions ({and_count}). Use periods or semicolons.")
    
    return issues

def rule_based_fix(text):
    result = text
    for phrase, replacement in WORDY_PHRASES.items():
        if replacement == "(remove)":
            result = re.sub(re.escape(phrase) + r'\s*', '', result, flags=re.IGNORECASE)
        else:
            result = re.sub(re.escape(phrase), replacement, result, flags=re.IGNORECASE)
    return result

print("RULE-BASED ANALYSIS")
print("=" * 60)
for s in SAMPLES:
    issues = detect_issues(s["text"])
    fixed = rule_based_fix(s["text"])
    print(f"\nSample {s['id']}:")
    print(f"  Original ({len(s['text'].split())} words): {s['text'][:80]}...")
    for issue in issues: print(f"  {issue}")
    if fixed != s["text"]:
        print(f"  Rule-fixed ({len(fixed.split())} words): {fixed[:80]}...")

## 8. Layer 2: LLM-Based Improvement

In [ ]:
IMPROVE_SYSTEM = """You are a professional editor who improves writing for clarity and concision.

Rules:
- Fix grammar errors
- Replace passive voice with active voice
- Remove wordy phrases
- Split long sentences
- Keep the SAME meaning
- Make every word earn its place
- Return ONLY the improved text"""

print("LLM IMPROVEMENT")
print("=" * 60)
llm_results = {}
for s in SAMPLES:
    improved = ask(IMPROVE_SYSTEM, f"Improve this text:\n{s['text']}")
    llm_results[s["id"]] = improved
    orig_words = len(s["text"].split())
    new_words = len(improved.split())
    reduction = round((1 - new_words / orig_words) * 100)
    print(f"\nSample {s['id']} ({reduction}% shorter):")
    print(f"  BEFORE: {s['text']}")
    print(f"  AFTER:  {improved}")

## 9. Layer 3: Combined Pipeline

Rule-based detection feeds into LLM improvement with specific guidance.

In [ ]:
GUIDED_SYSTEM = """You are a writing editor. Fix the specific issues listed below.

Rules:
- Address each listed issue
- Keep the same meaning
- Return ONLY the improved text"""

print("COMBINED PIPELINE")
print("=" * 60)
for s in SAMPLES[:3]:
    # Step 1: Detect issues
    issues = detect_issues(s["text"])
    
    # Step 2: LLM fix with issue guidance
    issue_list = "\n".join(issues) if issues else "General improvement needed."
    improved = ask(GUIDED_SYSTEM, f"Text:\n{s['text']}\n\nIssues found:\n{issue_list}")
    
    print(f"\nSample {s['id']}:")
    print(f"  Issues: {len(issues)}")
    print(f"  BEFORE: {s['text'][:100]}...")
    print(f"  AFTER:  {improved[:100]}...")

## 10. Readability Scoring

In [ ]:
def readability_metrics(text):
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    words = text.split()
    avg_sentence_len = len(words) / max(len(sentences), 1)
    passive_count = len(re.findall(r'\b(was|were|been|being|is|are)\s+\w+ed\b', text, re.IGNORECASE))
    wordy_count = sum(1 for p in WORDY_PHRASES if p.lower() in text.lower())
    return {
        "words": len(words),
        "sentences": len(sentences),
        "avg_sentence_len": round(avg_sentence_len, 1),
        "passive_voice": passive_count,
        "wordy_phrases": wordy_count,
    }

print("READABILITY COMPARISON")
print("=" * 70)
print(f"{'Sample':>8} {'Metric':<20} {'Before':>10} {'After':>10} {'Change':>10}")
print("-" * 70)

for s in SAMPLES:
    if s["id"] not in llm_results: continue
    before = readability_metrics(s["text"])
    after = readability_metrics(llm_results[s["id"]])
    
    for metric in ["words", "avg_sentence_len", "passive_voice", "wordy_phrases"]:
        b, a = before[metric], after[metric]
        change = a - b
        arrow = "\u2b07\ufe0f" if change < 0 else "\u2b06\ufe0f" if change > 0 else "\u2796"
        print(f"{s['id']:>8} {metric:<20} {b:>10} {a:>10} {arrow}{abs(change):>8}")
    print()

## 11. Improvement Modes

Different improvement goals for different contexts.

In [ ]:
MODES = {
    "Concision": "Make this text as concise as possible. Cut every unnecessary word. Same meaning, fewer words.",
    "Clarity": "Make this text as clear as possible. Simple words, short sentences. A 12-year-old should understand it.",
    "Professional polish": "Make this text polished and professional. Fix grammar, improve flow, keep it formal.",
    "Technical precision": "Make this text technically precise. Remove ambiguity. Every word should be exact.",
}

test_text = SAMPLES[0]["text"]
print(f"ORIGINAL: {test_text}\n")

for mode, instruction in MODES.items():
    result = ask(f"{instruction} Return ONLY the improved text.", f"Text:\n{test_text}")
    print(f"{mode} ({len(result.split())} words):")
    print(f"  {result}\n")

## 12. Batch Improvement with Explanation

In [ ]:
EXPLAIN_SYSTEM = """You are a writing coach. Improve the text AND explain each change.

Return JSON:
{
  "improved": "the improved text",
  "changes": [
    {"original": "original phrase", "replacement": "new phrase", "reason": "why"}
  ]
}"""

test = SAMPLES[3]["text"]  # The bureaucratic one
resp = ask(EXPLAIN_SYSTEM, f"Text:\n{test}")
text = clean(resp)
if "```" in text: text = re.sub(r"```(?:json)?\s*", "", text).replace("```", "")
s, e = text.find("{"), text.rfind("}") + 1
if s >= 0 and e > s:
    try:
        result = json.loads(text[s:e])
        print(f"BEFORE: {test}")
        print(f"\nAFTER: {result.get('improved', 'N/A')}")
        print(f"\nCHANGES:")
        for c in result.get("changes", []):
            if isinstance(c, dict):
                print(f"  \u2022 \"{c.get('original','')}\" \u2192 \"{c.get('replacement','')}\"")
                print(f"    Reason: {c.get('reason','')}")
    except:
        print(f"Response: {resp[:300]}")

## 13. Edge Cases

In [ ]:
# Already-good text should not be changed much
good_text = "The team shipped the feature on Friday. Users adopted it quickly. Revenue increased 12% in the first week."
improved = ask(IMPROVE_SYSTEM, f"Improve this text:\n{good_text}")
print("EDGE CASE: Already-good text")
print(f"  Original:  {good_text}")
print(f"  Improved:  {improved}")
print(f"  \u2192 Should be nearly identical\n")

# Text with intentional passive voice (scientific)
scientific = "The samples were incubated at 37\u00b0C for 24 hours. The results were measured using spectrophotometry."
improved2 = ask(IMPROVE_SYSTEM, f"Improve this text:\n{scientific}")
print("EDGE CASE: Scientific passive voice")
print(f"  Original:  {scientific}")
print(f"  Improved:  {improved2}")
print(f"  \u2192 Scientific writing often uses passive voice intentionally")

## 14. Common Mistakes & Key Takeaways

| Mistake | Fix |
|---------|-----|
| Removing all passive voice | Some contexts (scientific, legal) use it intentionally |
| Over-simplifying | Technical precision matters in some contexts |
| No explanation of changes | Users learn more when changes are explained |
| Rule-based only | Rules catch patterns but miss context; LLMs fill the gap |

| # | Takeaway |
|---|----------|
| 1 | **Rule-based detection** catches known patterns reliably and fast |
| 2 | **LLM rewriting** handles context-dependent improvements |
| 3 | **Combined pipeline** (detect \u2192 guide \u2192 fix) outperforms either alone |
| 4 | **Readability metrics** make improvement measurable (word count, sentence length, passive count) |
| 5 | **Multiple modes** (concision, clarity, polish) serve different writing goals |
| 6 | **Good text should not change** \u2014 the improver must know when to leave things alone |
| 7 | **Explained changes** are more useful than silent rewrites |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*